# Phase 1: Data Pipeline
## SVG Scaling Laws — CS-GY 6923 Optional Project

This notebook orchestrates all Phase 1 steps end-to-end:
1. Mount Google Drive & clone repo
2. Install dependencies
3. Run `01_download_data.py`
4. Run `02_clean_normalize.py`
5. Run `03_train_tokenizer.py`
6. Run `04_prepare_dataset.py`
7. Generate visualizations
8. Validate final dataset

**Expected runtime:** ~30-45 minutes (mostly data download + tokenization)

**Runtime recommendation:** GPU is not needed for Phase 1, but using Colab Pro is recommended for the RAM.

---
## Cell 0: Mount Google Drive
All outputs will be saved to Drive so they persist across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/svg-scaling-laws'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Project dir: {DRIVE_DIR}')

---
## Cell 1: Clone / Pull Repository
**Option A** (first time): Clone from GitHub  
**Option B** (resuming): Pull latest changes

Edit the repo URL below before running.

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/svg-scaling-laws.git'  # <-- EDIT THIS
REPO_DIR = '/content/svg-scaling-laws'

import os
if os.path.exists(REPO_DIR):
    print('Repo already exists, pulling latest ...')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning repo ...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

---
## Cell 2: Install Dependencies

In [ ]:
# Install all requirements
!pip install -q -r requirements.txt

# cairosvg needs system libraries
!apt-get install -q -y libcairo2-dev libpango1.0-dev libgdk-pixbuf2.0-dev libffi-dev shared-mime-info
!pip install -q cairosvg

print('All dependencies installed.')

---
## Cell 3: Configure Output Paths
We symlink `outputs/` to Google Drive so everything persists.

In [ ]:
import os
import yaml

REPO_DIR = '/content/svg-scaling-laws'
DRIVE_OUTPUTS = '/content/drive/MyDrive/svg-scaling-laws/outputs'
LOCAL_OUTPUTS = os.path.join(REPO_DIR, 'outputs')

os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

# Create symlink from repo/outputs → Drive/outputs
if os.path.exists(LOCAL_OUTPUTS) and not os.path.islink(LOCAL_OUTPUTS):
    import shutil
    # Move any existing local outputs to Drive first
    if os.listdir(LOCAL_OUTPUTS):
        print('Moving existing local outputs to Drive ...')
        shutil.copytree(LOCAL_OUTPUTS, DRIVE_OUTPUTS, dirs_exist_ok=True)
    shutil.rmtree(LOCAL_OUTPUTS)

if not os.path.islink(LOCAL_OUTPUTS):
    os.symlink(DRIVE_OUTPUTS, LOCAL_OUTPUTS)
    print(f'Created symlink: {LOCAL_OUTPUTS} → {DRIVE_OUTPUTS}')
else:
    print(f'Symlink already exists: {LOCAL_OUTPUTS} → {os.readlink(LOCAL_OUTPUTS)}')

# Create required subdirectories
for subdir in ['data/raw', 'data/cleaned', 'data/binary', 'tokenizer', 
               'checkpoints', 'logs', 'samples', 'plots', 'report']:
    os.makedirs(os.path.join(DRIVE_OUTPUTS, subdir), exist_ok=True)

print('Directory structure ready.')
!ls {DRIVE_OUTPUTS}

---
## Cell 4: Step 1 — Download Datasets
Downloads from HuggingFace. The primary dataset is `starvector/svg-icons-simple` (~89k SVGs).  
If we need more tokens to reach 100M, it automatically fetches supplementary datasets.

**Expected time:** 5-15 minutes (depends on HuggingFace bandwidth)

In [ ]:
import shutil, os

DRIVE_OUTPUTS = '/content/drive/MyDrive/svg-scaling-laws/outputs'

dirs_to_wipe = [
    'data/raw',
    'data/cleaned',
    'data/binary',
    'tokenizer',
]

print("Wiping cached intermediate outputs ...")
for d in dirs_to_wipe:
    full = os.path.join(DRIVE_OUTPUTS, d)
    if os.path.exists(full):
        shutil.rmtree(full)
        os.makedirs(full)
        print(f"  Cleared: {full}")
    else:
        os.makedirs(full, exist_ok=True)
        print(f"  Created: {full}")

print("\nAll intermediate outputs cleared. Ready for fresh run.")

In [ ]:
import subprocess, sys

print("=" * 60)
print("STEP 1: Download datasets (full re-download, no cache)")
print("=" * 60)
!python scripts/01_download_data.py --config configs/data_config.yaml

# Verify fonts_simple.jsonl line count
import os
fonts_path = 'outputs/data/raw/fonts_simple.jsonl'
if os.path.exists(fonts_path):
    result = subprocess.run(['wc', '-l', fonts_path], capture_output=True, text=True)
    n_lines = int(result.stdout.strip().split()[0])
    size_mb = os.path.getsize(fonts_path) / 1e6
    expected_min = 900_000   # at 0.65 fraction of ~1.23M = ~800k, be conservative
    status = "✓" if n_lines >= expected_min else f"✗ EXPECTED >={expected_min:,}"
    print(f"\nfonts_simple.jsonl: {n_lines:,} lines  ({size_mb:.0f} MB)  {status}")
else:
    print("✗ fonts_simple.jsonl not found!")

print("\n" + "=" * 60)
print("STEP 2: Clean and normalize")
print("=" * 60)
!python scripts/02_clean_normalize.py --config configs/data_config.yaml

print("\n" + "=" * 60)
print("STEP 3: Train tokenizer")
print("=" * 60)
!python scripts/03_train_tokenizer.py --config configs/data_config.yaml

print("\n" + "=" * 60)
print("STEP 4: Prepare binary dataset")
print("=" * 60)
!python scripts/04_prepare_dataset.py --config configs/data_config.yaml

In [ ]:
# Verify download
import json
manifest_path = 'outputs/data/raw/download_manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)
print('Download manifest:')
print(json.dumps(manifest, indent=2))

---
## Cell 5: Step 2 — Clean & Normalize SVGs
Runs the 9-step cleaning pipeline on all downloaded SVGs.

**Expected time:** 5-10 minutes

In [ ]:
!python scripts/02_clean_normalize.py --config configs/data_config.yaml

In [ ]:
# Verify cleaning stats
with open('outputs/data/dataset_stats.json') as f:
    stats = json.load(f)

print('Cleaning statistics:')
print(f"  Total input SVGs:  {stats.get('total_input', '?'):>10,}")
print(f"  Total kept SVGs:   {stats.get('total_output', '?'):>10,}")
print(f"  Removed breakdown: {stats.get('removed', {})}")

# Count lines in cleaned.jsonl
import subprocess as sp
result = sp.run(['wc', '-l', 'outputs/data/cleaned/cleaned.jsonl'], capture_output=True, text=True)
print(f"\n  cleaned.jsonl lines: {result.stdout.strip()}")

---
## Cell 6: Step 3 — Train BPE Tokenizer
Trains a 4096-vocab BPE tokenizer on the cleaned SVG corpus.  
Also generates the token frequency distribution plot.

**Expected time:** 5-10 minutes

In [ ]:
!python scripts/03_train_tokenizer.py --config configs/data_config.yaml

In [ ]:
# Display the generated plots
from IPython.display import Image, display
import os

for plot_name in ['token_freq_distribution.png', 'sequence_length_histogram.png']:
    plot_path = f'outputs/plots/{plot_name}'
    if os.path.exists(plot_path):
        print(f'\n--- {plot_name} ---')
        display(Image(plot_path))
    else:
        print(f'Plot not found: {plot_path}')

In [ ]:
# Show tokenizer stats
with open('outputs/tokenizer/tokenizer_stats.json') as f:
    tok_stats = json.load(f)

print(f"Vocabulary size: {tok_stats['vocab_size']}")
print(f"\nToken length statistics:")
ls = tok_stats['length_stats']
print(f"  Mean:    {ls['mean']:.1f} tokens")
print(f"  Median:  {ls['median']:.1f} tokens")
print(f"  P95:     {ls['p95']:.1f} tokens")
print(f"  P99:     {ls['p99']:.1f} tokens")
print(f"  Max:     {ls['max']} tokens")
print(f"  > 1024:  {ls['exceeds_max_length']} SVGs ({ls['exceeds_max_length_pct']:.1f}%)")

print(f"\nTop 10 tokens by frequency:")
for entry in tok_stats['top_30_tokens'][:10]:
    print(f"  id={entry['id']:>5}  {entry['token']:<30}  count={entry['count']:,}")

---
## Cell 7: Step 4 — Prepare Binary Dataset
Tokenizes all SVGs, applies length filter, splits 98/1/1 by file, and writes
train/val/test as numpy uint16 memmap files.

**Expected time:** 10-20 minutes

In [ ]:
!python scripts/04_prepare_dataset.py --config configs/data_config.yaml

In [ ]:
# Verify binary outputs
import numpy as np

with open('outputs/data/binary/split_info.json') as f:
    split_info = json.load(f)

print('Split information:')
print(json.dumps(split_info, indent=2))

# Quick binary load check
print('\nVerifying binary files ...')
for split in ['train', 'val', 'test']:
    arr = np.memmap(f'outputs/data/binary/{split}.bin', dtype=np.uint16, mode='r')
    expected = split_info[f'{split}_tokens']
    status = '✓' if len(arr) == expected else f'✗ (got {len(arr)}, expected {expected})'
    size_mb = arr.nbytes / 1e6
    print(f"  {split:5s}.bin:  {len(arr):>12,} tokens  {size_mb:>8.1f} MB  {status}")

TARGET = 100_000_000
train_tokens = split_info['train_tokens']
check = '✓' if train_tokens >= TARGET else '✗ BELOW TARGET'
print(f'\nTraining tokens: {train_tokens:,}  {check}  (target: {TARGET:,})')

---
## Cell 8: Visualize Sample SVGs
Render 3 SVGs at different complexity levels (simple / medium / complex)  
to confirm the data looks correct.

In [ ]:
import json
import sys
sys.path.insert(0, '/content/svg-scaling-laws')

from src.tokenizer_utils import load_tokenizer, encode as tok_encode
from src.svg_utils import render_svg_to_png
from IPython.display import Image, display
import io

tokenizer = load_tokenizer('outputs/tokenizer')

# Load all cleaned SVGs
all_svgs = []
with open('outputs/data/cleaned/cleaned.jsonl') as f:
    for line in f:
        if line.strip():
            all_svgs.append(json.loads(line)['svg'])

# Compute lengths and pick representative examples
import random
random.seed(42)
sample = random.sample(all_svgs, min(5000, len(all_svgs)))
lengths = [(len(tok_encode(tokenizer, s)), s) for s in sample]
lengths.sort(key=lambda x: x[0])

n = len(lengths)
examples = {
    'Simple (~50 tokens)':  lengths[n // 20],      # ~5th percentile
    'Medium (~200 tokens)': lengths[n // 2],        # ~50th percentile
    'Complex (~800 tokens)': lengths[min(n-1, int(n*0.95))],  # ~95th percentile
}

for label, (n_tokens, svg) in examples.items():
    print(f'\n=== {label} — {n_tokens} tokens, {len(svg)} chars ===')
    print('SVG code (first 300 chars):')
    print(svg[:300])
    png = render_svg_to_png(svg, output_size=200)
    if png:
        display(Image(data=png))
    else:
        print('(Rendering failed — cairosvg may not support this SVG feature)')

---
## Cell 9: Full Dataset Statistics Summary
Compile and display all Phase 1 statistics for the report.

In [ ]:
import json

with open('outputs/data/dataset_stats.json') as f:
    stats = json.load(f)

print('=' * 60)
print('PHASE 1 FINAL STATISTICS')
print('=' * 60)

print('\n--- Cleaning ---')
print(f"  Files input:          {stats.get('total_input', '?'):>10,}")
print(f"  Files kept:           {stats.get('total_output', '?'):>10,}")
removed = stats.get('removed', {})
for reason, count in removed.items():
    print(f"  Removed ({reason:<18}): {count:>6,}")

if 'tokenizer' in stats:
    print('\n--- Tokenizer ---')
    tok = stats['tokenizer']
    ls = tok['length_stats']
    print(f"  Vocabulary size:      {tok['vocab_size']:>10,}")
    print(f"  Mean seq length:      {ls['mean']:>10.1f} tokens")
    print(f"  Median seq length:    {ls['median']:>10.1f} tokens")
    print(f"  P95 seq length:       {ls['p95']:>10.1f} tokens")
    print(f"  P99 seq length:       {ls['p99']:>10.1f} tokens")

if 'splits' in stats:
    print('\n--- Splits ---')
    sp = stats['splits']
    print(f"  Train SVGs:           {sp['train_svgs']:>10,}")
    print(f"  Val SVGs:             {sp['val_svgs']:>10,}")
    print(f"  Test SVGs:            {sp['test_svgs']:>10,}")
    print(f"  Train tokens:         {sp['train_tokens']:>10,}")
    print(f"  Val tokens:           {sp['val_tokens']:>10,}")
    print(f"  Test tokens:          {sp['test_tokens']:>10,}")
    print(f"  Total tokens:         {sp['total_tokens']:>10,}")

TARGET = 100_000_000
train_tokens = stats.get('splits', {}).get('train_tokens', 0)
print(f"\n{'='*60}")
if train_tokens >= TARGET:
    print(f"✓ Training tokens: {train_tokens:,} (above 100M target)")
    print("Phase 1 COMPLETE. Proceed to Phase 2 (notebook 02_scaling_study.ipynb).")
else:
    print(f"✗ Training tokens: {train_tokens:,} (BELOW 100M target)")
    print("Need more data. See Section 4.1 of blueprint for supplementary datasets.")
print('=' * 60)

---
## Cell 10: Save All Outputs to Drive
Since we symlinked `outputs/` to Drive, everything is already persisted.  
This cell just confirms what's saved.

In [ ]:
import os
DRIVE_OUTPUTS = '/content/drive/MyDrive/svg-scaling-laws/outputs'

print('Files saved to Drive:')
for root, dirs, files in os.walk(DRIVE_OUTPUTS):
    # Skip huge binary files in listing
    dirs[:] = [d for d in dirs if d not in ['raw']]
    level = root.replace(DRIVE_OUTPUTS, '').count(os.sep)
    indent = ' ' * 2 * level
    folder = os.path.basename(root)
    print(f'{indent}{folder}/')
    for fname in files:
        fpath = os.path.join(root, fname)
        size = os.path.getsize(fpath)
        subindent = ' ' * 2 * (level + 1)
        if size > 1e6:
            print(f'{subindent}{fname}  ({size/1e6:.1f} MB)')
        else:
            print(f'{subindent}{fname}  ({size/1e3:.1f} KB)')

print('\nAll Phase 1 outputs are persisted to Google Drive.')
print('You can safely disconnect and resume in a new Colab session.')

---
## Done!

Phase 1 is complete. Here's what was produced:

| File | Purpose |
|------|---------|
| `outputs/data/raw/*.jsonl` | Raw SVGs from HuggingFace |
| `outputs/data/cleaned/cleaned.jsonl` | Cleaned SVGs (after 9-step pipeline) |
| `outputs/tokenizer/tokenizer.json` | Trained BPE tokenizer |
| `outputs/data/binary/train.bin` | Training tokens (uint16 memmap) |
| `outputs/data/binary/val.bin` | Validation tokens |
| `outputs/data/binary/test.bin` | Test tokens |
| `outputs/data/binary/split_info.json` | Split sizes and metadata |
| `outputs/data/dataset_stats.json` | Full pipeline statistics |
| `outputs/plots/token_freq_distribution.png` | Zipf plot for report |
| `outputs/plots/sequence_length_histogram.png` | Length distribution for report |

**Next:** Open `notebooks/02_scaling_study.ipynb` to run Phase 2 (model training).